# 04 dataset joining corrected




In [1]:
# import tools for selecting the correct python executable
import os
import sys

# use the same python executable for spark driver and workers
python_path = sys.executable

os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

from pyspark.sql import SparkSession

# create or reuse the spark session
spark = (
    SparkSession.builder
    .appName("dataset_joining_corrected")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "Europe/London")
    .config("spark.pyspark.python", python_path)
    .config("spark.pyspark.driver.python", python_path)
    .config("spark.executorEnv.PYSPARK_PYTHON", python_path)
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("python version:", sys.version.split()[0])
print("python path:", python_path)
print("pyspark version:", spark.version)
print(
    "shuffle partitions:",
    spark.conf.get("spark.sql.shuffle.partitions")
)
print(
    "session time zone:",
    spark.conf.get("spark.sql.session.timeZone")
)
print("spark ui:", spark.sparkContext.uiWebUrl)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/14 22:35:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/14 22:35:32 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/07/14 22:35:32 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/07/14 22:35:32 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


python version: 3.11.15
python path: /opt/anaconda3/envs/pyspark35/bin/python
pyspark version: 3.5.1
shuffle partitions: 4
session time zone: Europe/London
spark ui: http://192.168.1.8:4043


In [2]:
# import dataframe and path tools
from pathlib import Path
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# identify the project root
current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

timetable_path = (
    project_root
    / "data"
    / "processed"
    / "timetable"
)

vehicle_path = (
    project_root
    / "data"
    / "processed"
    / "vehicle_locations"
)

joined_path = (
    project_root
    / "data"
    / "processed"
    / "joined_transport_data"
)

joined_csv_path = (
    project_root
    / "data"
    / "processed"
    / "joined_transport_data_csv"
)

print("project root:", project_root)
print("timetable path exists:", timetable_path.exists())
print("vehicle path exists:", vehicle_path.exists())
print("joined parquet path:", joined_path)
print("joined csv path:", joined_csv_path)


NameError: name 'spark' is not defined

## load the processed datasets

In [3]:
# load the processed datasets
timetable_df = spark.read.parquet(
    str(timetable_path)
)

vehicle_df = spark.read.parquet(
    str(vehicle_path)
)

timetable_rows = timetable_df.count()
vehicle_rows = vehicle_df.count()

print(
    "timetable rows:",
    f"{timetable_rows:,}"
)

print(
    "vehicle observations:",
    f"{vehicle_rows:,}"
)

print(
    "timetable partitions:",
    timetable_df.rdd.getNumPartitions()
)

print(
    "vehicle partitions:",
    vehicle_df.rdd.getNumPartitions()
)


timetable rows: 5,370,199
vehicle observations: 9,546
timetable partitions: 8
vehicle partitions: 4


In [4]:
# validate the columns required for joining
required_timetable_columns = [
    "agency_noc",
    "route_id",
    "route_short_name",
    "service_id",
    "trip_id",
    "vehicle_journey_code",
    "trip_headsign",
    "direction_id",
    "stop_id",
    "stop_sequence",
    "arrival_seconds",
    "departure_seconds"
]

required_vehicle_columns = [
    "source_file",
    "recorded_at_time",
    "operator_ref",
    "line_ref",
    "published_line_name",
    "dated_vehicle_journey_ref",
    "direction_ref",
    "vehicle_ref",
    "origin_aimed_departure_time"
]

missing_timetable_columns = [
    column_name
    for column_name in required_timetable_columns
    if column_name not in timetable_df.columns
]

missing_vehicle_columns = [
    column_name
    for column_name in required_vehicle_columns
    if column_name not in vehicle_df.columns
]

print(
    "missing timetable columns:",
    missing_timetable_columns
)

print(
    "missing vehicle columns:",
    missing_vehicle_columns
)

if missing_timetable_columns:
    raise ValueError(
        f"missing timetable columns {missing_timetable_columns}"
    )

if missing_vehicle_columns:
    raise ValueError(
        f"missing vehicle columns {missing_vehicle_columns}"
    )

print("all required joining columns are available")


missing timetable columns: []
missing vehicle columns: []
all required joining columns are available


## create one timetable row for each trip

In [5]:
# group stop level records into one row for each timetable trip
trip_level_df = (
    timetable_df
    .groupBy(
        "agency_noc",
        "route_id",
        "route_short_name",
        "service_id",
        "trip_id",
        "vehicle_journey_code",
        "trip_headsign",
        "direction_id"
    )
    .agg(
        F.min(
            "departure_seconds"
        ).alias(
            "scheduled_start_seconds"
        ),
        F.max(
            "arrival_seconds"
        ).alias(
            "scheduled_end_seconds"
        ),
        F.min(
            "stop_sequence"
        ).alias(
            "first_stop_sequence"
        ),
        F.max(
            "stop_sequence"
        ).alias(
            "last_stop_sequence"
        ),
        F.countDistinct(
            "stop_id"
        ).alias(
            "scheduled_stop_count"
        )
    )
)

print(
    "trip level timetable rows:",
    f"{trip_level_df.count():,}"
)

trip_level_df.show(
    10,
    truncate=False
)


trip level timetable rows: 135,003


26/07/14 22:36:09 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/14 22:36:09 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/14 22:36:09 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/14 22:36:09 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/14 22:36:11 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/14 22:36:11 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/14 22:36:11 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/14 22:36:11 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/14 22:36:11 WARN RowBasedKeyValueBatch: Calling spill() on

+----------+--------+----------------+----------+------------------------------------------+--------------------+------------------------------+------------+-----------------------+---------------------+-------------------+------------------+--------------------+
|agency_noc|route_id|route_short_name|service_id|trip_id                                   |vehicle_journey_code|trip_headsign                 |direction_id|scheduled_start_seconds|scheduled_end_seconds|first_stop_sequence|last_stop_sequence|scheduled_stop_count|
+----------+--------+----------------+----------+------------------------------------------+--------------------+------------------------------+------------+-----------------------+---------------------+-------------------+------------------+--------------------+
|TNXB      |1719805 |997             |620       |VJ006ad9cca356b4b9020bb8f1d151560e105c5585|VJ22                |Walsall Bus Station           |0           |38880                  |42780                |0    

## create compatible joining keys

In [6]:
# create a compact uppercase key
def compact_key(column_name):
    return F.upper(
        F.regexp_replace(
            F.trim(
                F.coalesce(
                    F.col(column_name).cast("string"),
                    F.lit("")
                )
            ),
            r"[^A-Za-z0-9]",
            ""
        )
    )


# extract the final numeric journey value
# examples vj133 and vj_0133 become 133
def journey_number_key(column_name):
    raw_value = F.upper(
        F.trim(
            F.coalesce(
                F.col(column_name).cast("string"),
                F.lit("")
            )
        )
    )

    numeric_value = F.regexp_extract(
        raw_value,
        r"([0-9]+)$",
        1
    )

    return F.when(
        numeric_value != "",
        numeric_value.cast("long").cast("string")
    )


# map live direction labels to gtfs style values
# direction is used only for ranking and is not required for a match
def vehicle_direction_key(column_name):
    direction_value = F.lower(
        F.trim(
            F.coalesce(
                F.col(column_name).cast("string"),
                F.lit("")
            )
        )
    )

    return (
        F.when(
            direction_value.isin(
                "outbound",
                "out",
                "0"
            ),
            F.lit("0")
        )
        .when(
            direction_value.isin(
                "inbound",
                "in",
                "1"
            ),
            F.lit("1")
        )
    )


print("joining key functions created")


joining key functions created


In [7]:
# prepare timetable keys
trip_prepared_df = (
    trip_level_df
    .withColumn(
        "trip_operator_key",
        compact_key("agency_noc")
    )
    .withColumn(
        "trip_line_key",
        compact_key("route_short_name")
    )
    .withColumn(
        "trip_journey_number_key",
        journey_number_key(
            "vehicle_journey_code"
        )
    )
    .withColumn(
        "trip_direction_key",
        compact_key("direction_id")
    )
    .withColumn(
        "scheduled_clock_seconds",
        F.pmod(
            F.col("scheduled_start_seconds"),
            F.lit(86400)
        )
    )
    .repartition(
        4,
        "trip_operator_key",
        "trip_line_key"
    )
)

trip_prepared_df.cache()

print(
    "prepared timetable trips:",
    f"{trip_prepared_df.count():,}"
)


26/07/14 22:36:29 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/14 22:36:29 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/14 22:36:29 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/14 22:36:29 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/14 22:36:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/14 22:36:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/14 22:36:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/14 22:36:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/14 22:36:30 WARN RowBasedKeyValueBatch: Calling spill() on

prepared timetable trips: 135,003


In [8]:
# prepare vehicle keys
line_ref_key = compact_key("line_ref")
published_line_key = compact_key(
    "published_line_name"
)

vehicle_prepared_df = (
    vehicle_df
    .withColumn(
        "vehicle_operator_key",
        compact_key("operator_ref")
    )
    .withColumn(
        "vehicle_line_key",
        F.when(
            line_ref_key != "",
            line_ref_key
        ).otherwise(
            published_line_key
        )
    )
    .withColumn(
        "vehicle_journey_number_key",
        journey_number_key(
            "dated_vehicle_journey_ref"
        )
    )
    .withColumn(
        "vehicle_direction_key",
        vehicle_direction_key(
            "direction_ref"
        )
    )
    .withColumn(
        "aimed_start_seconds",
        (
            F.hour(
                F.col(
                    "origin_aimed_departure_time"
                )
            ) * 3600
            +
            F.minute(
                F.col(
                    "origin_aimed_departure_time"
                )
            ) * 60
            +
            F.second(
                F.col(
                    "origin_aimed_departure_time"
                )
            )
        )
    )
    .withColumn(
        "observation_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(
                    F.col("source_file"),
                    F.lit("")
                ),
                F.coalesce(
                    F.col("vehicle_ref"),
                    F.lit("")
                ),
                F.coalesce(
                    F.col(
                        "recorded_at_time"
                    ).cast("string"),
                    F.lit("")
                ),
                F.coalesce(
                    F.col("line_ref"),
                    F.lit("")
                )
            ),
            256
        )
    )
    .repartition(
        4,
        "vehicle_operator_key",
        "vehicle_line_key"
    )
)

vehicle_prepared_df.cache()

print(
    "prepared vehicle observations:",
    f"{vehicle_prepared_df.count():,}"
)


26/07/14 22:36:46 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 39:>                                                         (0 + 4) / 4]

prepared vehicle observations: 9,546


## inspect the corrected keys

In [9]:
# inspect timetable journey keys
print("timetable journey keys")

trip_prepared_df.select(
    "agency_noc",
    "route_short_name",
    "vehicle_journey_code",
    "trip_journey_number_key",
    "direction_id",
    "scheduled_clock_seconds"
).where(
    F.col(
        "trip_journey_number_key"
    ).isNotNull()
).dropDuplicates().show(
    20,
    truncate=False
)

# inspect live journey keys
print("vehicle journey keys")

vehicle_prepared_df.select(
    "operator_ref",
    "line_ref",
    "dated_vehicle_journey_ref",
    "vehicle_journey_number_key",
    "direction_ref",
    "aimed_start_seconds"
).where(
    F.col(
        "vehicle_journey_number_key"
    ).isNotNull()
).dropDuplicates().show(
    20,
    truncate=False
)


timetable journey keys


+----------+----------------+--------------------+-----------------------+------------+-----------------------+
|agency_noc|route_short_name|vehicle_journey_code|trip_journey_number_key|direction_id|scheduled_clock_seconds|
+----------+----------------+--------------------+-----------------------+------------+-----------------------+
|TBTN      |X38             |VJ107               |107                    |0           |67800                  |
|TNXB      |4M              |VJ525               |525                    |1           |48900                  |
|AMNO      |7               |vj_8                |8                      |0           |45000                  |
|AMNO      |7               |vj_22               |22                     |1           |44280                  |
|AMNO      |7               |vj_22               |22                     |0           |61800                  |
|AMNO      |7               |vj_23               |23                     |0           |63600            

In [10]:
# measure meaningful overlap using combined keys
timetable_route_keys = (
    trip_prepared_df
    .select(
        "trip_operator_key",
        "trip_line_key"
    )
    .filter(
        (F.col("trip_operator_key") != "")
        &
        (F.col("trip_line_key") != "")
    )
    .distinct()
)

vehicle_route_keys = (
    vehicle_prepared_df
    .select(
        F.col(
            "vehicle_operator_key"
        ).alias(
            "trip_operator_key"
        ),
        F.col(
            "vehicle_line_key"
        ).alias(
            "trip_line_key"
        )
    )
    .filter(
        (F.col("trip_operator_key") != "")
        &
        (F.col("trip_line_key") != "")
    )
    .distinct()
)

route_key_overlap = (
    timetable_route_keys
    .join(
        vehicle_route_keys,
        on=[
            "trip_operator_key",
            "trip_line_key"
        ],
        how="inner"
    )
    .count()
)

timetable_exact_keys = (
    trip_prepared_df
    .select(
        "trip_operator_key",
        "trip_line_key",
        "trip_journey_number_key"
    )
    .filter(
        F.col(
            "trip_journey_number_key"
        ).isNotNull()
    )
    .distinct()
)

vehicle_exact_keys = (
    vehicle_prepared_df
    .select(
        F.col(
            "vehicle_operator_key"
        ).alias(
            "trip_operator_key"
        ),
        F.col(
            "vehicle_line_key"
        ).alias(
            "trip_line_key"
        ),
        F.col(
            "vehicle_journey_number_key"
        ).alias(
            "trip_journey_number_key"
        )
    )
    .filter(
        F.col(
            "trip_journey_number_key"
        ).isNotNull()
    )
    .distinct()
)

exact_key_overlap = (
    timetable_exact_keys
    .join(
        vehicle_exact_keys,
        on=[
            "trip_operator_key",
            "trip_line_key",
            "trip_journey_number_key"
        ],
        how="inner"
    )
    .count()
)

print(
    "operator and line overlap:",
    route_key_overlap
)

print(
    "operator line and numeric journey overlap:",
    exact_key_overlap
)


operator and line overlap: 154
operator line and numeric journey overlap: 4351


## stage one exact journey matching



In [11]:
# create aliases for exact journey matching
vehicle_alias = vehicle_prepared_df.alias("v")
trip_alias = trip_prepared_df.alias("t")

exact_raw_time_difference = F.abs(
    F.col("t.scheduled_clock_seconds")
    -
    F.col("v.aimed_start_seconds")
)

exact_time_difference = F.least(
    exact_raw_time_difference,
    F.lit(86400)
    -
    exact_raw_time_difference
)

exact_join_condition = (
    F.col(
        "v.vehicle_journey_number_key"
    ).isNotNull()
    &
    (
        F.col(
            "v.vehicle_operator_key"
        )
        ==
        F.col(
            "t.trip_operator_key"
        )
    )
    &
    (
        F.col(
            "v.vehicle_line_key"
        )
        ==
        F.col(
            "t.trip_line_key"
        )
    )
    &
    (
        F.col(
            "v.vehicle_journey_number_key"
        )
        ==
        F.col(
            "t.trip_journey_number_key"
        )
    )
)

exact_candidates_df = (
    vehicle_alias
    .join(
        trip_alias,
        on=exact_join_condition,
        how="inner"
    )
    .select(
        F.col(
            "v.observation_id"
        ).alias(
            "observation_id"
        ),
        F.col(
            "t.trip_id"
        ).alias(
            "matched_trip_id"
        ),
        F.col(
            "t.service_id"
        ).alias(
            "matched_service_id"
        ),
        F.col(
            "t.route_id"
        ).alias(
            "matched_route_id"
        ),
        F.col(
            "t.route_short_name"
        ).alias(
            "matched_route_short_name"
        ),
        F.col(
            "t.vehicle_journey_code"
        ).alias(
            "matched_vehicle_journey_code"
        ),
        F.col(
            "t.direction_id"
        ).alias(
            "matched_direction_id"
        ),
        F.col(
            "t.trip_headsign"
        ).alias(
            "matched_trip_headsign"
        ),
        F.col(
            "t.scheduled_start_seconds"
        ).alias(
            "scheduled_start_seconds"
        ),
        F.col(
            "t.scheduled_clock_seconds"
        ).alias(
            "scheduled_clock_seconds"
        ),
        F.col(
            "t.scheduled_end_seconds"
        ).alias(
            "scheduled_end_seconds"
        ),
        F.col(
            "t.scheduled_stop_count"
        ).alias(
            "scheduled_stop_count"
        ),
        exact_time_difference.alias(
            "schedule_time_difference_seconds"
        ),
        F.when(
            F.col(
                "v.vehicle_direction_key"
            )
            ==
            F.col(
                "t.trip_direction_key"
            ),
            1
        ).otherwise(
            0
        ).alias(
            "direction_key_match"
        ),
        F.lit(
            "exact_operator_line_journey"
        ).alias(
            "match_method"
        )
    )
)

print(
    "exact candidate rows:",
    f"{exact_candidates_df.count():,}"
)


exact candidate rows: 15,629


In [12]:
# select one exact timetable trip for each observation
exact_window = (
    Window
    .partitionBy(
        "observation_id"
    )
    .orderBy(
        F.col(
            "direction_key_match"
        ).desc(),
        F.col(
            "schedule_time_difference_seconds"
        ).asc_nulls_last(),
        F.col(
            "matched_trip_id"
        ).asc()
    )
)

exact_best_matches_df = (
    exact_candidates_df
    .withColumn(
        "match_rank",
        F.row_number().over(
            exact_window
        )
    )
    .filter(
        F.col("match_rank") == 1
    )
    .drop("match_rank")
)

print(
    "observations with an exact match:",
    f"{exact_best_matches_df.count():,}"
)


observations with an exact match: 7,178


## stage two route and time fallback




In [13]:
# choose the maximum allowed time difference for a fallback match
fallback_window_seconds = 1800

# keep only observations that did not receive an exact match
unmatched_after_exact_df = (
    vehicle_prepared_df
    .join(
        exact_best_matches_df.select(
            "observation_id"
        ),
        on="observation_id",
        how="left_anti"
    )
)

print(
    "observations remaining after exact matching:",
    f"{unmatched_after_exact_df.count():,}"
)


observations remaining after exact matching: 2,368


In [14]:
# create aliases for fallback matching
fallback_vehicle_alias = (
    unmatched_after_exact_df.alias("v")
)

fallback_trip_alias = (
    trip_prepared_df.alias("t")
)

fallback_raw_time_difference = F.abs(
    F.col("t.scheduled_clock_seconds")
    -
    F.col("v.aimed_start_seconds")
)

fallback_time_difference = F.least(
    fallback_raw_time_difference,
    F.lit(86400)
    -
    fallback_raw_time_difference
)

fallback_join_condition = (
    F.col(
        "v.aimed_start_seconds"
    ).isNotNull()
    &
    (
        F.col(
            "v.vehicle_operator_key"
        )
        ==
        F.col(
            "t.trip_operator_key"
        )
    )
    &
    (
        F.col(
            "v.vehicle_line_key"
        )
        ==
        F.col(
            "t.trip_line_key"
        )
    )
    &
    (
        fallback_time_difference
        <=
        F.lit(
            fallback_window_seconds
        )
    )
)

fallback_candidates_df = (
    fallback_vehicle_alias
    .join(
        fallback_trip_alias,
        on=fallback_join_condition,
        how="inner"
    )
    .select(
        F.col(
            "v.observation_id"
        ).alias(
            "observation_id"
        ),
        F.col(
            "t.trip_id"
        ).alias(
            "matched_trip_id"
        ),
        F.col(
            "t.service_id"
        ).alias(
            "matched_service_id"
        ),
        F.col(
            "t.route_id"
        ).alias(
            "matched_route_id"
        ),
        F.col(
            "t.route_short_name"
        ).alias(
            "matched_route_short_name"
        ),
        F.col(
            "t.vehicle_journey_code"
        ).alias(
            "matched_vehicle_journey_code"
        ),
        F.col(
            "t.direction_id"
        ).alias(
            "matched_direction_id"
        ),
        F.col(
            "t.trip_headsign"
        ).alias(
            "matched_trip_headsign"
        ),
        F.col(
            "t.scheduled_start_seconds"
        ).alias(
            "scheduled_start_seconds"
        ),
        F.col(
            "t.scheduled_clock_seconds"
        ).alias(
            "scheduled_clock_seconds"
        ),
        F.col(
            "t.scheduled_end_seconds"
        ).alias(
            "scheduled_end_seconds"
        ),
        F.col(
            "t.scheduled_stop_count"
        ).alias(
            "scheduled_stop_count"
        ),
        fallback_time_difference.alias(
            "schedule_time_difference_seconds"
        ),
        F.when(
            F.col(
                "v.vehicle_direction_key"
            )
            ==
            F.col(
                "t.trip_direction_key"
            ),
            1
        ).otherwise(
            0
        ).alias(
            "direction_key_match"
        ),
        F.lit(
            "fallback_operator_line_time"
        ).alias(
            "match_method"
        )
    )
)

print(
    "fallback candidate rows:",
    f"{fallback_candidates_df.count():,}"
)


fallback candidate rows: 112,001


In [15]:
# select one fallback timetable trip for each remaining observation
fallback_window = (
    Window
    .partitionBy(
        "observation_id"
    )
    .orderBy(
        F.col(
            "direction_key_match"
        ).desc(),
        F.col(
            "schedule_time_difference_seconds"
        ).asc(),
        F.col(
            "matched_trip_id"
        ).asc()
    )
)

fallback_best_matches_df = (
    fallback_candidates_df
    .withColumn(
        "match_rank",
        F.row_number().over(
            fallback_window
        )
    )
    .filter(
        F.col("match_rank") == 1
    )
    .drop("match_rank")
)

print(
    "observations with a fallback match:",
    f"{fallback_best_matches_df.count():,}"
)


[Stage 175:>                                                        (0 + 4) / 4]

observations with a fallback match: 2,368


## combine matches and preserve every observation

In [16]:
# combine exact and fallback matches
best_matches_df = (
    exact_best_matches_df
    .unionByName(
        fallback_best_matches_df
    )
)

# confirm that one observation does not have multiple final matches
duplicate_match_ids = (
    best_matches_df
    .groupBy(
        "observation_id"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

print(
    "duplicate final match ids:",
    duplicate_match_ids
)

if duplicate_match_ids > 0:
    raise ValueError(
        "more than one final match exists for an observation"
    )

# attach the selected match to every live observation
joined_df = (
    vehicle_prepared_df
    .join(
        best_matches_df,
        on="observation_id",
        how="left"
    )
)

print(
    "joined observation rows:",
    f"{joined_df.count():,}"
)


duplicate final match ids: 0
joined observation rows: 9,546


In [17]:
# create transparent match status and quality values
joined_df = (
    joined_df
    .withColumn(
        "is_timetable_matched",
        F.when(
            F.col(
                "matched_trip_id"
            ).isNotNull(),
            1
        ).otherwise(
            0
        )
    )
    .withColumn(
        "match_quality",
        F.when(
            F.col(
                "matched_trip_id"
            ).isNull(),
            "unmatched"
        )
        .when(
            (
                F.col(
                    "match_method"
                )
                ==
                "exact_operator_line_journey"
            )
            &
            (
                F.col(
                    "schedule_time_difference_seconds"
                )
                <=
                900
            ),
            "high"
        )
        .when(
            F.col(
                "match_method"
            )
            ==
            "exact_operator_line_journey",
            "strong"
        )
        .when(
            (
                F.col(
                    "match_method"
                )
                ==
                "fallback_operator_line_time"
            )
            &
            (
                F.col(
                    "direction_key_match"
                )
                ==
                1
            )
            &
            (
                F.col(
                    "schedule_time_difference_seconds"
                )
                <=
                900
            ),
            "medium"
        )
        .otherwise(
            "provisional"
        )
    )
)

original_vehicle_count = (
    vehicle_prepared_df.count()
)

final_joined_count = (
    joined_df.count()
)

print(
    "original vehicle observations:",
    f"{original_vehicle_count:,}"
)

print(
    "final joined observations:",
    f"{final_joined_count:,}"
)

if final_joined_count != original_vehicle_count:
    raise ValueError(
        "final joined rows do not match vehicle observations"
    )

print("one final row exists for every vehicle observation")


original vehicle observations: 9,546
final joined observations: 9,546
one final row exists for every vehicle observation


## evaluate the corrected match result

In [18]:
# calculate match counts and match rate
matched_observations = (
    joined_df
    .filter(
        F.col(
            "is_timetable_matched"
        )
        ==
        1
    )
    .count()
)

unmatched_observations = (
    final_joined_count
    -
    matched_observations
)

match_rate = (
    matched_observations
    /
    final_joined_count
    *
    100
    if final_joined_count > 0
    else 0
)

print(
    "total vehicle observations:",
    f"{final_joined_count:,}"
)

print(
    "matched observations:",
    f"{matched_observations:,}"
)

print(
    "unmatched observations:",
    f"{unmatched_observations:,}"
)

print(
    "join match rate:",
    f"{match_rate:.2f}%"
)

print("match method summary")

(
    joined_df
    .groupBy(
        "match_method"
    )
    .count()
    .orderBy(
        F.desc("count")
    )
    .show(
        truncate=False
    )
)

print("match quality summary")

(
    joined_df
    .groupBy(
        "match_quality"
    )
    .count()
    .orderBy(
        F.desc("count")
    )
    .show(
        truncate=False
    )
)


total vehicle observations: 9,546
matched observations: 9,546
unmatched observations: 0
join match rate: 100.00%
match method summary
+---------------------------+-----+
|match_method               |count|
+---------------------------+-----+
|exact_operator_line_journey|7178 |
|fallback_operator_line_time|2368 |
+---------------------------+-----+

match quality summary
+-------------+-----+
|match_quality|count|
+-------------+-----+
|strong       |6743 |
|medium       |2364 |
|high         |435  |
|provisional  |4    |
+-------------+-----+



In [19]:
# inspect exact matches
print("sample exact matches")

joined_df.filter(
    F.col(
        "match_method"
    )
    ==
    "exact_operator_line_journey"
).select(
    "vehicle_ref",
    "operator_ref",
    "line_ref",
    "dated_vehicle_journey_ref",
    "origin_aimed_departure_time",
    "matched_trip_id",
    "matched_route_short_name",
    "matched_vehicle_journey_code",
    "matched_direction_id",
    "schedule_time_difference_seconds",
    "match_quality"
).show(
    20,
    truncate=False
)

# inspect fallback matches
print("sample fallback matches")

joined_df.filter(
    F.col(
        "match_method"
    )
    ==
    "fallback_operator_line_time"
).select(
    "vehicle_ref",
    "operator_ref",
    "line_ref",
    "dated_vehicle_journey_ref",
    "direction_ref",
    "origin_aimed_departure_time",
    "matched_trip_id",
    "matched_route_short_name",
    "matched_vehicle_journey_code",
    "matched_direction_id",
    "schedule_time_difference_seconds",
    "direction_key_match",
    "match_quality"
).show(
    20,
    truncate=False
)

# inspect unmatched observations
print("sample unmatched observations")

joined_df.filter(
    F.col(
        "is_timetable_matched"
    )
    ==
    0
).select(
    "source_file",
    "vehicle_ref",
    "operator_ref",
    "line_ref",
    "published_line_name",
    "dated_vehicle_journey_ref",
    "direction_ref",
    "origin_aimed_departure_time",
    "recorded_at_time"
).show(
    20,
    truncate=False
)


sample exact matches
+-----------+------------+--------+-------------------------+---------------------------+------------------------------------------+------------------------+----------------------------+--------------------+--------------------------------+-------------+
|vehicle_ref|operator_ref|line_ref|dated_vehicle_journey_ref|origin_aimed_departure_time|matched_trip_id                           |matched_route_short_name|matched_vehicle_journey_code|matched_direction_id|schedule_time_difference_seconds|match_quality|
+-----------+------------+--------+-------------------------+---------------------------+------------------------------------------+------------------------+----------------------------+--------------------+--------------------------------+-------------+
|E063       |TCVW        |X1      |139                      |2026-07-11 02:05:00        |VJ810c825fb4c93f97bb117d1fc4a977b16425ce4c|X1                      |VJ139                       |1                   |5340   

## prepare the joined dataset for later notebooks

In [20]:
# repartition and cache the joined dataset
joined_df = (
    joined_df
    .repartition(
        4,
        "line_ref"
    )
)

joined_df.cache()

joined_count = joined_df.count()

print(
    "final joined rows:",
    f"{joined_count:,}"
)

print(
    "joined partitions:",
    joined_df.rdd.getNumPartitions()
)

joined_df.createOrReplaceTempView(
    "joined_transport_data"
)

print("temporary sql view created")


final joined rows: 9,546
joined partitions: 4
temporary sql view created


In [21]:
# calculate route level matching results
join_line_summary = spark.sql(
    '''
    SELECT
        operator_ref,
        line_ref,
        published_line_name,
        COUNT(*) AS total_observations,
        SUM(is_timetable_matched) AS matched_observations,
        ROUND(
            SUM(is_timetable_matched) * 100.0
            / COUNT(*),
            2
        ) AS match_rate_percent
    FROM joined_transport_data
    GROUP BY
        operator_ref,
        line_ref,
        published_line_name
    ORDER BY
        total_observations DESC
    LIMIT 30
    '''
)

join_line_summary.show(
    truncate=False
)


+------------+--------+-------------------+------------------+--------------------+------------------+
|operator_ref|line_ref|published_line_name|total_observations|matched_observations|match_rate_percent|
+------------+--------+-------------------+------------------+--------------------+------------------+
|TNXB        |6       |6                  |296               |296                 |100.00            |
|TNXB        |11A     |11A                |250               |250                 |100.00            |
|TNXB        |4       |4                  |235               |235                 |100.00            |
|TNXB        |74      |74                 |234               |234                 |100.00            |
|TNXB        |11C     |11C                |203               |203                 |100.00            |
|TNXB        |9       |9                  |186               |186                 |100.00            |
|TNXB        |16      |16                 |179               |179        

In [22]:
# display the spark execution plan
joined_df.explain(
    mode="formatted"
)


== Physical Plan ==
AdaptiveSparkPlan (102)
+- == Final Plan ==
   TableCacheQueryStage (101), Statistics(sizeInBytes=4.6 MiB, rowCount=9.55E+3)
   +- InMemoryTableScan (1)
         +- InMemoryRelation (2)
               +- AdaptiveSparkPlan (100)
                  +- Exchange (99)
                     +- Project (98)
                        +- SortMergeJoin LeftOuter (97)
                           :- Sort (10)
                           :  +- Exchange (9)
                           :     +- InMemoryTableScan (3)
                           :           +- InMemoryRelation (4)
                           :                 +- AdaptiveSparkPlan (8)
                           :                    +- Exchange (7)
                           :                       +- Project (6)
                           :                          +- Scan parquet  (5)
                           +- Sort (96)
                              +- Exchange (95)
                                 +- Union (94)
        

## save and verify the corrected parquet dataset

In [23]:
# create the processed parent folder when needed
joined_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

# save the joined dataset as parquet
(
    joined_df.write
    .mode("overwrite")
    .parquet(
        str(joined_path)
    )
)

print("joined transport data saved")
print("parquet location:", joined_path)

# read and verify the saved parquet data
saved_joined_df = spark.read.parquet(
    str(joined_path)
)

saved_joined_count = (
    saved_joined_df.count()
)

print(
    "original joined rows:",
    f"{joined_count:,}"
)

print(
    "saved joined rows:",
    f"{saved_joined_count:,}"
)

print(
    "saved partitions:",
    saved_joined_df.rdd.getNumPartitions()
)

if saved_joined_count != joined_count:
    raise ValueError(
        "saved joined count does not match original count"
    )

print("joined parquet verification successful")


joined transport data saved
parquet location: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/processed/joined_transport_data
original joined rows: 9,546
saved joined rows: 9,546
saved partitions: 4
joined parquet verification successful


## optional csv export

parquet remains the main project format


In [24]:
# export one csv part file for simple inspection
(
    saved_joined_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option(
        "header",
        True
    )
    .csv(
        str(joined_csv_path)
    )
)

print("joined csv dataset saved")
print("csv location:", joined_csv_path)


[Stage 545:>                                                        (0 + 1) / 1]

joined csv dataset saved
csv location: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/processed/joined_transport_data_csv


In [25]:
# display the final corrected joining summary
print("corrected dataset joining summary")
print("-" * 50)

print(
    "trip level timetable rows:",
    f"{trip_level_df.count():,}"
)

print(
    "vehicle observations:",
    f"{original_vehicle_count:,}"
)

print(
    "matched observations:",
    f"{matched_observations:,}"
)

print(
    "unmatched observations:",
    f"{unmatched_observations:,}"
)

print(
    "join match rate:",
    f"{match_rate:.2f}%"
)

print(
    "fallback time window seconds:",
    fallback_window_seconds
)

print(
    "final partitions:",
    joined_df.rdd.getNumPartitions()
)

print("main saved format: parquet")
print("parquet location:", joined_path)
print("csv location:", joined_csv_path)


corrected dataset joining summary
--------------------------------------------------


[Stage 546:=================================================>       (7 + 1) / 8]

trip level timetable rows: 135,003
vehicle observations: 9,546
matched observations: 9,546
unmatched observations: 0
join match rate: 100.00%
fallback time window seconds: 1800
final partitions: 4
main saved format: parquet
parquet location: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/processed/joined_transport_data
csv location: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/processed/joined_transport_data_csv


In [ ]:


from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# use the same python for spark driver and workers
python_path = sys.executable

os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

# create or reuse the spark session
spark = (
    SparkSession.builder
    .appName("joined_data_validation")
    .config(
        "spark.sql.shuffle.partitions",
        "4"
    )
    .config(
        "spark.sql.session.timeZone",
        "Europe/London"
    )
    .config(
        "spark.pyspark.python",
        python_path
    )
    .config(
        "spark.pyspark.driver.python",
        python_path
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# identify the project folder
current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

joined_path = (
    project_root
    / "data"
    / "processed"
    / "joined_transport_data"
)

# load the saved parquet dataset
joined_df = spark.read.parquet(
    str(joined_path)
)

print(
    "loaded joined rows:",
    f"{joined_df.count():,}"
)

print(
    "loaded partitions:",
    joined_df.rdd.getNumPartitions()
)

# show exact and fallback match results
(
    joined_df
    .groupBy(
        "match_method",
        "match_quality"
    )
    .count()
    .orderBy(
        F.desc("count")
    )
    .show(
        truncate=False
    )
)

[Stage 8:==============>                                            (1 + 3) / 4]